# Module 0: Kiến thức tiên quyết — code

Notebook đi kèm site *Tối ưu hóa phi tuyến & lồi*. Chạy tuần tự từng ô (Colab có sẵn numpy, scipy, sympy, cvxpy). Tự giải tay trước rồi đối chiếu kết quả.

## Ma trận 2×2: det, nghịch đảo, giá trị riêng (dạng A)

Kiểm bảng đại số tuyến tính.

In [1]:
import numpy as np
A = np.array([[2, 1], [1, 3]])
print("det =", round(np.linalg.det(A), 6))
print("A^-1 * 5 =\n", np.round(np.linalg.inv(A) * 5, 6))
print("giá trị riêng =", np.round(np.linalg.eigvalsh(A), 4))

det = 5.0
A^-1 * 5 =
 [[ 3. -1.]
 [-1.  2.]]
giá trị riêng = [1.382 3.618]


## Điểm dừng và phân loại bằng sympy (dạng B)

f = x³ − 3x + y²: giải ∇f = 0 rồi xét Hessian từng điểm.

In [2]:
import sympy as sp
x, y = sp.symbols("x y", real=True)
f = x**3 - 3*x + y**2
H = sp.hessian(f, (x, y))
for s in sp.solve([sp.diff(f, x), sp.diff(f, y)], (x, y), dict=True):
    print(s, "f =", f.subs(s), "eig H =", list(H.subs(s).eigenvals()))

{x: -1, y: 0} f = 2 eig H = [-6, 2]
{x: 1, y: 0} f = -2 eig H = [6, 2]


## Taylor bậc hai (dạng C)

f = eˣ + xy tại gốc, d = (0.1, 0.2).

In [3]:
import numpy as np
g = np.array([1.0, 0.0]); H = np.array([[1.0, 1.0], [1.0, 0.0]]); d = np.array([0.1, 0.2])
approx = 1 + g @ d + 0.5 * d @ H @ d
true = np.exp(0.1) + 0.1 * 0.2
print("xấp xỉ =", approx, " thật =", round(true, 5))

xấp xỉ = 1.125  thật = 1.12517


## LP và đối ngẫu bằng scipy (dạng D)

max 3x1+2x2 với hai ràng buộc ≤; so giá trị gốc và đối ngẫu.

In [4]:
from scipy.optimize import linprog
A = [[1, 1], [1, 3]]; b = [4, 6]; c = [3, 2]
p = linprog([-v for v in c], A_ub=A, b_ub=b)
d = linprog(b, A_ub=[[-1, -1], [-1, -3]], b_ub=[-3, -2])
print("gốc  x =", p.x, " giá trị =", -p.fun)
print("đối ngẫu y =", d.x, " giá trị =", d.fun)

gốc  x = [4. 0.]  giá trị = 12.0
đối ngẫu y = [3. 0.]  giá trị = 12.0


## Bài code · Gradient descent

Đầu vào: $f=x^2+3y^2$, điểm đầu $(4,2)$, bước $\alpha=0.1$. Đầu ra: điểm sau 50 vòng và $\lVert\nabla f\rVert$.

In [5]:
import numpy as np
grad = lambda p: np.array([2*p[0], 6*p[1]])
p = np.array([4., 2.])
for _ in range(50): p = p - 0.1 * grad(p)
print("p =", p, " |grad| =", np.linalg.norm(grad(p)))

p = [5.70899077e-05 2.53530120e-20]  |grad| = 0.00011417981541647683


## Bài code · Kiểm gradient bằng sai phân

Đầu vào: $f=e^{x}+xy$, điểm $(0.3,0.5)$. Đầu ra: gradient giải tích và sai phân trung tâm, sai số lớn nhất.

In [6]:
import numpy as np
f = lambda p: np.exp(p[0]) + p[0]*p[1]
g = lambda p: np.array([np.exp(p[0]) + p[1], p[0]])
p = np.array([0.3, 0.5]); h = 1e-6
num = np.array([(f(p + h*e) - f(p - h*e)) / (2*h) for e in np.eye(2)])
print(g(p), num, "sai số:", abs(g(p) - num).max())

[1.84985881 0.3       ] [1.84985881 0.3       ] sai số: 1.0820633278285641e-10


## Bài code · Bình phương tối thiểu

Đầu vào: $A=\begin{pmatrix}1&1\\1&2\\1&3\end{pmatrix}$, $b=(1,2,2)$. Đầu ra: nghiệm của $\min\lVert Ax-b\rVert^2$ bằng phương trình chuẩn và lstsq.

In [7]:
import numpy as np
A = np.array([[1, 1], [1, 2], [1, 3.]]); b = np.array([1, 2, 2.])
print(np.linalg.solve(A.T @ A, A.T @ b), np.linalg.lstsq(A, b, rcond=None)[0])

[0.66666667 0.5       ] [0.66666667 0.5       ]


## Bài code · Có cực tiểu không? (coercive)

Đầu ra: thử $e^x$ và $x^2$ trên lưới lớn; nhận xét giá trị nhỏ nhất ở biên lưới.

In [8]:
import numpy as np
x = np.linspace(-20, 20, 400001)
for name, f in (("e^x", np.exp), ("x^2", lambda t: t**2)):
    v = f(x); print(name, "min trên lưới =", v.min(), "tại x =", x[v.argmin()])

e^x

 min trên lưới = 2.061153622438558e-09 tại x = -20.0
x^2 min trên lưới = 0.0 tại x = 0.0
